In [2]:
from datetime import date, timedelta
import pandas as pd
import json
import os

### Pollutants JSON

In [11]:
CSV_POLEN = r"..\new_datasets\datos_gramineas.csv"
df = pd.read_csv(CSV_POLEN)
today = date.today().strftime('%Y-%m-%d')

df_today = df[df['fecha'] == today]

clean_names = {
    "NO2 (ug/m3)": "NO2",
    "O3 (ug/m3)": "O3",
    "PM10 (ug/m3)": "PM10",
    "PM2.5 (ug/m3)": "PM2.5",
    "CO (mg/m3)": "CO",
    "SO2 (ug/m3)": "SO2"
}
datos = df_today[list(clean_names.keys())].iloc[0].to_dict()
pollutants_json = {clean_names[k]: round(float(v), 1) for k, v in datos.items()}

folder_path = r"..\..\Prediction-service"
file_name = "pollutants.json"
full_path = os.path.join(folder_path, file_name)

os.makedirs(folder_path, exist_ok=True)
with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(pollutants_json, f, indent=4, ensure_ascii=False)


### Polen Prediction JSON

In [12]:
tipos_polinicos = {
    "Gramineas": "gramineas",
    "Cupresacea": "cupresaceas",
    "Olivo": "olivo",
    "Platano_de_paseo": "platano",
    "Urticaceas": "urticaceas",
    "Quenopodiaceas": "quenopodiaceas"
}

folder_path = r"..\..\Prediction-service"
file_name = "polen.json"
full_path = os.path.join(folder_path, file_name)
with open(full_path, 'r', encoding='utf-8') as f:
    json_polen = json.load(f)

for jsonName, tipo in tipos_polinicos.items():
    CSV_POLEN = rf"..\new_datasets\datos_{tipo}.csv"
    df = pd.read_csv(CSV_POLEN)
    today = date.today().strftime('%Y-%m-%d')

    today_dt = pd.to_datetime(date.today())
    tomorrow_dt = today_dt + timedelta(days=1)
    after_tomorrow_dt = today_dt + timedelta(days=2)

    df['fecha'] = pd.to_datetime(df['fecha'])
    today_dt = pd.to_datetime(date.today())
    df_lastWeek = df[df['fecha'] < today_dt].tail(7)
    historical_data = df_lastWeek['granos_de_polen_x_metro_cubico'].tolist()

    def get_val(df, fecha):
        try:
            valor = df.loc[fecha, 'granos_de_polen_x_metro_cubico']
            if pd.isna(valor):
                return 0.0
            else:
                return float(valor.iloc[0]) if hasattr(valor, 'iloc') else float(valor)
        except Exception:
            return 0.0

    df = df.set_index('fecha')
    data = {
        "historical": [round(v, 1) for v in historical_data],
        "prediction": {
            "today": round(get_val(df, today_dt), 1),
            "tomorrow": round(get_val(df, tomorrow_dt), 1),
            "day_after_tomorrow": round(get_val(df, after_tomorrow_dt), 1)
        }
    }
    json_polen[jsonName] = data

with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(json_polen, f, indent=4, ensure_ascii=False)